# Panel Data — Wages, Experience & Union Membership

| | |
|---|---|
| **Outcome** | `lwage` — log hourly wage |
| **Panel** | 595 individuals × 7 years (1976–1982) |
| **Dataset** | Cornwell & Rupert (1988) via R `plm` package |

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

In [13]:
df = pd.read_csv(Path("../data/raw/wages.csv"))

# Reconstruct individual ID and time period from sequential row index
df['id']   = (df['rownames'] - 1) // 7 + 1   # 1–595
df['year'] = (df['rownames'] - 1) % 7 + 1976  # 1976–1982

# Encode yes/no strings as 0/1
for col in ['bluecol', 'south', 'smsa', 'married', 'union', 'black']:
    df[col] = (df[col] == 'yes').astype(int)
df['female'] = (df['sex'] == 'female').astype(int)
df = df.drop(columns=['rownames', 'sex'])

# Rename columns to human-readable names
df = df.rename(columns={
    'lwage':   'log_wage',
    'exp':     'experience',
    'wks':     'weeks_worked',
    'union':   'union_member',
    'ed':      'education',
    'black':   'is_black',
    'female':  'is_female',
    'bluecol': 'is_bluecol',
    'south':   'lives_south',
    'smsa':    'lives_metro',
    'married': 'is_married',
    'ind':     'industry',
})

N = df['id'].nunique()
T = df['year'].nunique()

print(f"Individuals (N) : {N}")
print(f"Time periods (T): {T}")
print(f"Total obs (N×T) : {len(df)}")
df.head(14)

Individuals (N) : 595
Time periods (T): 7
Total obs (N×T) : 4165


,experience,weeks_worked,is_bluecol,industry,lives_south,lives_metro,is_married,union_member,education,is_black,log_wage,id,year,is_female
0,3,32,0,0,1,0,1,0,9,0,5.56068,1,1976,0
1,4,43,0,0,1,0,1,0,9,0,5.72031,1,1977,0
2,5,40,0,0,1,0,1,0,9,0,5.99645,1,1978,0
3,6,39,0,0,1,0,1,0,9,0,5.99645,1,1979,0
4,7,42,0,1,1,0,1,0,9,0,6.06146,1,1980,0
5,8,35,0,1,1,0,1,0,9,0,6.17379,1,1981,0
6,9,32,0,1,1,0,1,0,9,0,6.24417,1,1982,0
7,30,34,1,0,0,0,1,0,11,0,6.16331,2,1976,0
8,31,27,1,0,0,0,1,0,11,0,6.21461,2,1977,0
9,32,33,1,1,0,0,1,1,11,0,6.26340,2,1978,0


In [20]:

VARS = ['log_wage', 'experience', 'weeks_worked', 'union_member', 'education', 'is_black', 'is_female']


In [21]:
overall_std = df[VARS].std()
between_std = df.groupby('id')[VARS].mean().std()
within_std  = df.groupby('id')[VARS].transform(lambda x: x - x.mean()).std()

pd.DataFrame({
    'Overall SD':  overall_std,
    'Between SD':  between_std,
    'Within SD':   within_std,
    'Within %':   (within_std / overall_std * 100).round(1),
}).round(3)

,Overall SD,Between SD,Within SD,Within %
log_wage,0.462,0.394,0.240,52.1
experience,10.966,10.790,2.000,18.2
weeks_worked,5.129,3.284,3.942,76.9
union_member,0.481,0.454,0.159,33.1
education,2.788,2.790,0.000,0.0
is_black,0.259,0.259,0.000,0.0
is_female,0.316,0.316,0.000,0.0


## Pooled OLS

Treats all 4,165 rows as independent observations — ignores the panel structure entirely.
Uses both within-person and between-person variation to estimate coefficients.

In [29]:
OUTCOME = 'log_wage'
REGRESSORS = ['experience', 'weeks_worked', 'union_member', 'education', 'is_black', 'is_female']

formula = f"{OUTCOME} ~ {' + '.join(REGRESSORS)}"


model = smf.ols(formula, data=df).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:               log_wage   R-squared:                       0.354
Model:                            OLS   Adj. R-squared:                  0.353
Method:                 Least Squares   F-statistic:                     380.0
Date:                Wed, 15 Jul 2026   Prob (F-statistic):               0.00
Time:                        05:33:03   Log-Likelihood:                -1778.3
No. Observations:                4165   AIC:                             3571.
Df Residuals:                    4158   BIC:                             3615.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
================================================================================
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept        5.1823      0.067     77.863      0.000       5.052       5.313
experience       0.0121      0.001     22.436      0.000       0.011       0.013
weeks_worked     0.0056      0.001      4.937      0.000       0.003       0.008
union_member     0.0996      0.013      7.837      0.000       0.075       0.124
education        0.0784      0.002     35.501      0.000       0.074       0.083
is_black        -0.1655      0.023     -7.223      0.000      -0.210      -0.121
is_female       -0.3817      0.019    -20.141      0.000      -0.419      -0.345
==============================================================================
Omnibus:                       56.961   Durbin-Watson:                   0.735
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              102.229
Skew:                          -0.043   Prob(JB):                     6.33e-23
Kurtosis:                       3.763   Cond. No.                         613.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [35]:
OUTCOME = 'log_wage'
REGRESSORS = ['experience', 'weeks_worked', 'union_member']

formula = f"{OUTCOME} ~ {' + '.join(REGRESSORS)} + C(id)"


model = smf.ols(formula, data=df).fit()
model.summary()

model.params[REGRESSORS]

experience      0.096892
weeks_worked    0.001104
union_member    0.031006
dtype: float64

In [40]:
REGRESSORS =  ['experience', 'weeks_worked', 'union_member', 'education', 'is_black', 'is_female']

formula = f"{OUTCOME} ~ {' + '.join(REGRESSORS)}"
model = smf.mixedlm(formula, data=df, groups=df['id']).fit()

In [41]:
model.params

Intercept        3.128669
experience       0.087309
weeks_worked     0.001162
union_member     0.036085
education        0.138750
is_black        -0.275690
is_female       -0.138774
Group Var       31.829463
dtype: float64

## Hausman Test

Tests whether FE and RE give statistically different answers on the time-varying coefficients.
If they do, RE's assumption is violated and FE is preferred.

In [46]:
from scipy.stats import chi2

# Variables both models estimated — time-varying only
COMMON_VARS = ['experience', 'weeks_worked', 'union_member']

# Grab FE and RE models (re-fit FE without clustering for correct covariance matrix)
fe_vars    = ['experience', 'weeks_worked', 'union_member']
fe_formula = f"{OUTCOME} ~ {' + '.join(fe_vars)} + C(id)"
fe_model   = smf.ols(fe_formula, data=df).fit()

re_vars    = ['experience', 'weeks_worked', 'union_member', 'education', 'is_black', 'is_female']
re_formula = f"{OUTCOME} ~ {' + '.join(re_vars)}"
re_model   = smf.mixedlm(re_formula, data=df, groups=df['id']).fit()

# Step 1: coefficient vectors for the common variables
b_fe = fe_model.params[COMMON_VARS].values
b_re = re_model.params[COMMON_VARS].values

# Step 2: covariance matrices for the common variables
fe_idx = [list(fe_model.params.index).index(v) for v in COMMON_VARS]
V_fe   = fe_model.cov_params().values[np.ix_(fe_idx, fe_idx)]

re_idx = [list(re_model.params.index).index(v) for v in COMMON_VARS]
V_re   = re_model.cov_params().values[np.ix_(re_idx, re_idx)]

# Step 3: Hausman statistic
diff   = b_fe - b_re
V_diff = V_fe - V_re
H      = diff @ np.linalg.pinv(V_diff) @ diff
p      = 1 - chi2.cdf(H, df=len(COMMON_VARS))

print(f"Hausman statistic : {H:.2f}")
print(f"Degrees of freedom: {len(COMMON_VARS)}")
print(f"p-value           : {p:.4f}")
print()
if p < 0.05:
    print("Reject RE — individual effects are correlated with regressors. Use Fixed Effects.")
else:
    print("Fail to reject RE — RE is consistent and more efficient. Use Random Effects.")

Hausman statistic : -382.86
Degrees of freedom: 3
p-value           : 1.0000

Fail to reject RE — RE is consistent and more efficient. Use Random Effects.
